In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
chip_data = pd.read_csv("../Data/chip_dataset_2026.csv")

# Attempt to parse the dates with a specified format first, then fall back to general parsing
chip_data['Release Year'] = pd.to_datetime(chip_data['Release Date'], format='%m/%d/%y', errors='coerce').dt.year

# For rows where parsing failed, attempt parsing again without a specific format
missing_years = chip_data['Release Year'].isna()
chip_data.loc[missing_years, 'Release Year'] = pd.to_datetime(
	chip_data.loc[missing_years, 'Release Date'], errors='coerce'
).dt.year
chip_data['Release Year'] = chip_data['Release Year'].astype('Int64')  # Use Int64 instead of int


In [3]:
# Coerce TDP to integers
chip_data['TDP (W)'] = pd.to_numeric(chip_data['TDP (W)'], errors='coerce').astype('Int64')

# Convert Process Size to numeric
chip_data['Process Size (nm)'] = pd.to_numeric(chip_data['Process Size (nm)'], errors='coerce')
chip_data['Die Size (mm^2)'] = pd.to_numeric(chip_data['Die Size (mm^2)'], errors='coerce')
chip_data['TDP (W)/Die Size (mm^2)'] = chip_data['TDP (W)'] / chip_data['Die Size (mm^2)']

# Convert Release Date to datetime for plotting
# chip_data['Release Date'] = pd.to_datetime(chip_data['Release Date'], errors='coerce')
chip_data['Release Year'] = pd.to_datetime(chip_data['Release Date'], format='%m/%d/%y', errors='coerce').dt.year
chip_data['Release Date'] = pd.to_datetime(chip_data['Release Date'], format='%m/%d/%y', errors='coerce')

In [4]:
chip_data.head()

,Product,Type,Release Date,Process Size (nm),TDP (W),Die Size (mm^2),Transistors (million),Freq (GHz),Foundry,Vendor,FP16 GFLOPS,FP32 GFLOPS,FP64 GFLOPS,Release Year,TDP (W)/Die Size (mm^2)
0,AMD Athlon 1000,CPU,2000-06-05,180.0,54,120.0,37,1000.0,NaN,AMD,NaN,NaN,NaN,2000.0,0.45
1,AMD Athlon 1000,CPU,2000-10-31,180.0,54,120.0,37,1000.0,NaN,AMD,NaN,NaN,NaN,2000.0,0.45
2,AMD Athlon 1100,CPU,2000-08-14,180.0,60,120.0,37,1100.0,NaN,AMD,NaN,NaN,NaN,2000.0,0.5
3,AMD Athlon 1133,CPU,2000-10-31,180.0,63,120.0,37,1133.0,NaN,AMD,NaN,NaN,NaN,2000.0,0.525
4,AMD Athlon 1200,CPU,2000-10-31,180.0,66,120.0,37,1200.0,NaN,AMD,NaN,NaN,NaN,2000.0,0.55


In [5]:
chip_data.columns

Index(['Product', 'Type', 'Release Date', 'Process Size (nm)', 'TDP (W)',
       'Die Size (mm^2)', 'Transistors (million)', 'Freq (GHz)', 'Foundry',
       'Vendor', 'FP16 GFLOPS', 'FP32 GFLOPS', 'FP64 GFLOPS', 'Release Year',
       'TDP (W)/Die Size (mm^2)'],
      dtype='object')

In [6]:
gpu_data = chip_data[chip_data['Type'] == 'GPU']
cpu_data = chip_data[chip_data['Type'] == 'CPU']
gpu_data = gpu_data.dropna(subset=['Release Year'])
gpu_data['FP32 GFLOPS/TDP (W)'] = gpu_data['FP32 GFLOPS'] / gpu_data['TDP (W)']
gpu_data = gpu_data.drop(gpu_data[gpu_data['Product']=="Intel Data Center GPU Max Subsystem"].index)

In [7]:

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Prepare data
plot_data = gpu_data.dropna(subset=['Release Date', 'Release Year', 'FP32 GFLOPS', 'TDP (W)', 'Process Size (nm)']).copy()



# Add trace for FP32 GFLOPS (left y-axis) - using circles
fig.add_trace(
    go.Scatter(
        x=plot_data['Release Date'], 
        y=plot_data['FP32 GFLOPS'], 
        name='FP32 GFLOPS',
        mode='markers',
        marker=dict(
            size=8,
            symbol='circle',  # Circle symbols
            color=plot_data['Process Size (nm)'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(
                title="Process Size (nm)<br>(FP32 GFLOPS)",
                x=1.15,
                len=0.4,
                y=0.75
            )
        )
    ),
    secondary_y=False,
)

# Add trace for TDP (right y-axis) - using diamonds
fig.add_trace(
    go.Scatter(
        x=plot_data['Release Date'], 
        y=plot_data['TDP (W)'], 
        name='TDP (W)',
        mode='markers',
        marker=dict(
            size=8,
            symbol='diamond',  # Diamond symbols
            color=plot_data['Process Size (nm)'],
            colorscale='Plasma',
            showscale=True,
            colorbar=dict(
                title="Process Size (nm)<br>(TDP)",
                x=1.15,
                len=0.4,
                y=0.25
            )
        )
    ),
    secondary_y=True,
)

# Update layout with black border
fig.update_layout(
    xaxis_title='Release Year',
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        x=1.25,
        y=0.5,
        xanchor='left',
        yanchor='middle'
    ),
    xaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True
    ),
    yaxis2=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True
    )
)

# Set y-axes titles
fig.update_yaxes(title_text='FP32 GFLOPS', secondary_y=False)
fig.update_yaxes(title_text='TDP (W)', secondary_y=True)

fig.show()


In [8]:

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Prepare data
plot_data = gpu_data.dropna(subset=['Release Date', 'Release Year', 'FP32 GFLOPS', 'TDP (W)', 'Process Size (nm)']).copy()

# Convert Release Date to datetime
plot_data['Release Date'] = pd.to_datetime(plot_data['Release Date'], errors='coerce')

# Add trace for FP32 GFLOPS (left y-axis) - using circles
fig.add_trace(
    go.Scatter(
        x=plot_data['Release Date'], 
        y=plot_data['FP32 GFLOPS'], 
        name='FP32 GFLOPS',
        mode='markers',
        marker=dict(
            size=8,
            symbol='circle',  # Circle symbols
            color=plot_data['Process Size (nm)'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(
                title="Process Size (nm)<br>(FP32 GFLOPS)",
                x=1.15,
                len=0.4,
                y=0.75
            )
        )
    ),
    secondary_y=False,
)

# Add trace for TDP (right y-axis) - using diamonds
fig.add_trace(
    go.Scatter(
        x=plot_data['Release Date'], 
        y=plot_data['TDP (W)'], 
        name='TDP (W)',
        mode='markers',
        marker=dict(
            size=8,
            symbol='diamond',  # Diamond symbols
            color=plot_data['Process Size (nm)'],
            colorscale='Plasma',
            showscale=True,
            colorbar=dict(
                title="Process Size (nm)<br>(TDP)",
                x=1.15,
                len=0.4,
                y=0.25
            )
        )
    ),
    secondary_y=True,
)

# Update layout with black border
fig.update_layout(
    xaxis_title='Release Year',
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        x=1.25,
        y=0.5,
        xanchor='left',
        yanchor='middle'
    ),
    xaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True,
        tickformat='%Y',  # Format ticks to show only the year
        dtick='M12'  # Show tick every 12 months (1 year)
    ),
    yaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True
    ),
    yaxis2=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True
    )
)

# Set y-axes titles
fig.update_yaxes(title_text='FP32 GFLOPS', secondary_y=False)
fig.update_yaxes(title_text='TDP (W)', secondary_y=True)

fig.show()


In [9]:
def plot_scatter(gpu_data, x, y, c, hover_info=None): 
    plot_data = gpu_data.dropna(subset=[x, y, c]).copy()

    plot_data[c] = pd.to_numeric(plot_data[c], errors='coerce')

    if hover_info is None:
        hover_info = plot_data['Product'].astype(str)
    else:
        hover_info = plot_data[hover_info].astype(str)

    fig = go.Figure()


    fig.add_trace(
        go.Scatter(
            x=plot_data[x], 
            y=plot_data[y], 
            mode='markers',
            marker=dict(
                size=8,
                color=plot_data[c],  # Color by column d
                colorscale='Viridis',  # Choose: 'Viridis', 'Plasma', 'Inferno', 'Magma', 'Cividis', 'Turbo', etc.
                showscale=True,
                colorbar=dict(
                    title=c,
                    thickness=15,
                    len=0.7
                )
            ),
            name='Data Points',
            # CHANGE 3: Add customdata and hovertemplate for custom hover info
            customdata=hover_info,
            hovertemplate='<b>%{customdata}</b><br>' +
                          x + ': %{x}<br>' +
                          y + ': %{y}<br>' +
                          '<extra></extra>'
        )
    )

    # Update layout with black borders
    fig.update_layout(
        height=500,
        width=600,
        # title='Scatter Plot',
        xaxis_title=x,
        yaxis_title=y,
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(
            showline=True,
            linewidth=1,
            linecolor='black',
            mirror=True,
            showgrid=True,
            gridcolor='lightgray',
            # range=[0, 1000]  # Set x-axis range [min, max]
        ),
        yaxis=dict(
            showline=True,
            linewidth=1,
            linecolor='black',
            mirror=True,
            showgrid=True,
            gridcolor='lightgray',
            # range=[0, 100000]  # Set y-axis range [min, max]
        ),
        # Optional: add a black border around the entire plot
        shapes=[dict(
            type='rect',
            x0=0, x1=1, y0=0, y1=1,
            xref='paper', yref='paper',
            line=dict(color='black', width=1)
        )]
    )

    fig.show()


In [10]:
x = 'TDP (W)'
y = 'FP32 GFLOPS'
c = 'Release Year'
plot_scatter(gpu_data, x, y, c)

In [11]:

x = 'TDP (W)'
y = 'FP32 GFLOPS'
c = 'Release Year'
plot_scatter(gpu_data, x, y, c)


In [12]:

x = 'Release Year'
y = 'FP32 GFLOPS/TDP (W)'
c = 'Process Size (nm)'
plot_scatter(gpu_data, x, y, c)

In [13]:

x = 'Die Size (mm^2)'
y = 'Process Size (nm)'
c = 'TDP (W)'
plot_scatter(gpu_data, x, y, c)


In [14]:

x = 'TDP (W)/Die Size (mm^2)'
y = 'FP32 GFLOPS'
c = 'Release Year'
plot_scatter(gpu_data, x, y, c)


In [15]:

x = 'TDP (W)'
y = 'Die Size (mm^2)'
c = 'Release Year'
plot_scatter(gpu_data, x, y, c)


In [16]:
x = 'Release Year'
y = 'Die Size (mm^2)'
c = 'TDP (W)'
plot_scatter(gpu_data, x, y, c)

In [17]:
x = 'Release Year'
y = 'TDP (W)'
c = 'Die Size (mm^2)'
plot_scatter(gpu_data, x, y, c)

In [18]:

x = 'TDP (W)/Die Size (mm^2)'
y = 'Die Size (mm^2)'
c = 'Release Year'
plot_scatter(gpu_data, x, y, c)